In [ ]:
#Data Loading
import pandas as pd
import numpy as np

df = pd.read_csv("rfm_modeling_snapshot.csv")  # or your merged dataset
``

In [ ]:
# Feature Prep + Leakage Check
# Drop leakage columns (example)
leak_cols = ['churn_next_60d_future_flag', 'post_snapshot_features']
df = df.drop(columns=[col for col in leak_cols if col in df.columns])

y = df['churn_next_60d']
X = df.drop(columns=['churn_next_60d'])

In [ ]:
#Train / Val / Test Split

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [ ]:
#Baseline Model (Logistic Regression)
from sklearn.linear_model import LogisticRegression

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)

In [ ]:
#Strong Model (Random Forest)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

In [ ]:
#Evaluation
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

In [ ]:
#Threshold Selection
threshold = 0.6
y_custom = (y_prob > threshold).astype(int)

In [ ]:
# Feature Importance
import matplotlib.pyplot as plt

feat_imp = pd.Series(model.feature_importances_, index=X.columns)
feat_imp.sort_values().tail(10).plot(kind='barh')
plt.show()

In [ ]:
#Save Model
import joblib
joblib.dump(model, "model.pkl")